# Maintain Agent — Cross-Session Intelligence & Hot-Memory

Tests LerimOAIAgent.maintain() end-to-end:
- Cross-session analysis (signal amplification, contradiction detection, gap detection)
- Memory consolidation, merging, archiving, decay
- Hot-memory curation (`.lerim/hot-memory.md`)

In [ ]:
import os, sys, json, tempfile
from pathlib import Path
from dotenv import load_dotenv
import nest_asyncio
nest_asyncio.apply()

load_dotenv(Path("/Users/kargarisaac/codes/personal/lerim/lerim-cli/.env"))
sys.path.insert(0, str(Path("/Users/kargarisaac/codes/personal/lerim/lerim-cli/.claude/worktrees/agent-af7d097c/src")))

from agents import set_tracing_disabled
set_tracing_disabled(disabled=True)

from lerim.runtime.oai_agent import LerimOAIAgent
from lerim.config.settings import get_config

print("All imports successful")
cfg = get_config()
print(f"Provider: {cfg.lead_role.provider}/{cfg.lead_role.model}")

## Test 1 — Maintain Prompt

In [ ]:
# Build a maintain prompt with test paths, print it, verify cross-session
# analysis and hot-memory sections are present.

from lerim.runtime.prompts.oai_maintain import (
	build_oai_maintain_prompt,
	build_oai_maintain_artifact_paths,
)

test_memory = Path("/tmp/lerim-maintain-test/memory")
test_run = Path("/tmp/lerim-maintain-test/workspace/maintain-test-001")
test_artifacts = build_oai_maintain_artifact_paths(test_run)

# Build prompt with some sample access stats
sample_stats = [
	{
		"memory_id": "20260301-use-postgresql",
		"last_accessed": "2026-03-20T10:00:00Z",
		"access_count": 8,
	},
	{
		"memory_id": "20260115-old-sqlite-note",
		"last_accessed": "2026-01-20T10:00:00Z",
		"access_count": 1,
	},
]

prompt = build_oai_maintain_prompt(
	memory_root=test_memory,
	run_folder=test_run,
	artifact_paths=test_artifacts,
	access_stats=sample_stats,
	decay_days=180,
	decay_archive_threshold=0.2,
	decay_min_confidence_floor=0.1,
	decay_recent_access_grace_days=30,
)

print(f"Prompt length: {len(prompt)} chars")
print("---")
print(prompt)
print("---")

# Verify cross-session analysis sections
assert "cross_session_analysis" in prompt, "cross_session_analysis checklist item missing"
assert "Signal Amplification" in prompt, "Signal Amplification section missing"
assert "Contradiction Detection" in prompt, "Contradiction Detection section missing"
assert "Gap Detection" in prompt, "Gap Detection section missing"

# Verify hot-memory sections
assert "curate_hot_memory" in prompt, "curate_hot_memory checklist item missing"
assert "hot-memory.md" in prompt, "hot-memory.md path missing"
assert "Hot Memory" in prompt, "Hot Memory format template missing"
assert "Active Decisions" in prompt, "Active Decisions section missing"
assert "Key Learnings" in prompt, "Key Learnings section missing"
assert "Watch Out" in prompt, "Watch Out section missing"
assert "~2000 tokens" in prompt, "Token budget missing"

# Verify standard maintain sections
assert "scan_memories_and_summaries" in prompt, "scan checklist missing"
assert "analyze_duplicates" in prompt, "analyze_duplicates missing"
assert "merge_similar" in prompt, "merge_similar missing"
assert "archive_low_value" in prompt, "archive_low_value missing"
assert "decay_check" in prompt, "decay_check missing"
assert "consolidate_related" in prompt, "consolidate_related missing"
assert "write_report" in prompt, "write_report missing"

# Verify decay stats are included
assert "20260301-use-postgresql" in prompt, "access stat missing"
assert "DECAY POLICY" in prompt, "decay policy missing"

# Verify codex-only (no explore/read/write tool references)
assert "codex" in prompt, "codex reference missing"
assert "explore()" not in prompt, "should not reference explore() tool"

print("\nAll prompt structure assertions passed")

## Test 2 — Full Maintain with Seed Memories

**NOTE:** This makes REAL LLM calls so it costs tokens and takes time.

In [ ]:
import tempfile
import logging
import frontmatter
from datetime import datetime, timezone

# Enable logging so we can see the agent's progress
logging.basicConfig(level=logging.INFO, format="%(name)s | %(message)s")

# ---------------------------------------------------------------
# 1. Create temp dirs mimicking .lerim structure
# ---------------------------------------------------------------
tmp_base = Path(tempfile.mkdtemp(prefix="lerim-maintain-"))
lerim_dir = tmp_base / ".lerim"
memory_root = lerim_dir / "memory"
workspace_root = lerim_dir / "workspace"

for sub in (
	memory_root / "decisions",
	memory_root / "learnings",
	memory_root / "summaries",
	memory_root / "archived" / "decisions",
	memory_root / "archived" / "learnings",
	workspace_root,
	lerim_dir / "index",
):
	sub.mkdir(parents=True, exist_ok=True)

print(f"Temp base:      {tmp_base}")
print(f".lerim dir:     {lerim_dir}")
print(f"Memory root:    {memory_root}")
print(f"Workspace root: {workspace_root}")

# ---------------------------------------------------------------
# 2. Seed 5 decision memories
# ---------------------------------------------------------------
decisions = [
	{
		"id": "20260301-use-postgresql",
		"title": "Use PostgreSQL for production database",
		"body": "PostgreSQL is the production database. It handles concurrent writes well, has excellent JSONB support for session data, and supports advanced indexing strategies. Chosen over MySQL for its extensibility and standards compliance.",
		"confidence": 0.95,
		"tags": ["database", "postgresql", "infrastructure"],
		"created": "2026-03-01T10:00:00Z",
		"updated": "2026-03-01T10:00:00Z",
		"source": "sync-20260301-100000-abc123",
	},
	{
		"id": "20260305-use-sqlite-local",
		"title": "Use SQLite for local storage",
		"body": "SQLite is used for local development and testing. It is lightweight and requires no server process. For production, consider a more robust database.",
		"confidence": 0.7,
		"tags": ["database", "sqlite", "local"],
		"created": "2026-03-05T14:00:00Z",
		"updated": "2026-03-05T14:00:00Z",
		"source": "sync-20260305-140000-def456",
	},
	{
		"id": "20260310-tabs-for-indentation",
		"title": "Use tabs for indentation",
		"body": "The project uses tabs for indentation in all code files. This is the user's explicit preference and must be followed consistently across all Python, JavaScript, and configuration files.",
		"confidence": 0.9,
		"tags": ["style", "formatting", "indentation"],
		"created": "2026-03-10T09:00:00Z",
		"updated": "2026-03-10T09:00:00Z",
		"source": "sync-20260310-090000-ghi789",
	},
	{
		"id": "20260312-two-space-indent",
		"title": "Use 2-space indentation",
		"body": "Some files might use 2-space indentation based on an earlier convention. This was mentioned once in passing.",
		"confidence": 0.4,
		"tags": ["style", "formatting", "indentation"],
		"created": "2026-03-12T11:00:00Z",
		"updated": "2026-03-12T11:00:00Z",
		"source": "sync-20260312-110000-jkl012",
	},
	{
		"id": "20260315-jwt-24h-expiry",
		"title": "JWT tokens with 24-hour expiry",
		"body": "Authentication uses JWT tokens with a 24-hour expiry window. Refresh tokens are issued alongside access tokens. Token rotation happens automatically on the client side.",
		"confidence": 0.85,
		"tags": ["auth", "jwt", "security"],
		"created": "2026-03-15T16:00:00Z",
		"updated": "2026-03-15T16:00:00Z",
		"source": "sync-20260315-160000-mno345",
	},
]

for d in decisions:
	fm_dict = {k: v for k, v in d.items() if k != "body"}
	post = frontmatter.Post(d["body"], **fm_dict)
	path = memory_root / "decisions" / f"{d['id']}.md"
	path.write_text(frontmatter.dumps(post) + "\n", encoding="utf-8")
	print(f"  Decision: {path.name}")

# ---------------------------------------------------------------
# 3. Seed 3 learning memories
# ---------------------------------------------------------------
learnings = [
	{
		"id": "20260302-connection-pooling-matters",
		"title": "Connection pooling matters for performance",
		"body": "Database connection pooling significantly improves performance under concurrent load. Without pooling, each request opens a new connection which is expensive. Use a pool size of 10-20 connections for typical workloads.",
		"kind": "insight",
		"confidence": 0.8,
		"tags": ["database", "performance", "connection-pooling"],
		"created": "2026-03-02T12:00:00Z",
		"updated": "2026-03-02T12:00:00Z",
		"source": "sync-20260302-120000-pqr678",
	},
	{
		"id": "20260308-connection-pool-sizing",
		"title": "Connection pool sizing guidelines",
		"body": "Connection pool size should be tuned based on workload. For CPU-bound apps, pool_size = num_cores * 2. For IO-bound apps, pool_size = num_cores * 4. Monitor active connections and adjust.",
		"kind": "procedure",
		"confidence": 0.75,
		"tags": ["database", "performance", "connection-pooling"],
		"created": "2026-03-08T15:00:00Z",
		"updated": "2026-03-08T15:00:00Z",
		"source": "sync-20260308-150000-stu901",
	},
	{
		"id": "20260318-never-mock-database",
		"title": "Never mock the database in tests",
		"body": "Database mocking hides real bugs like schema mismatches, constraint violations, and query performance issues. Always use a real test database (SQLite in-memory or Docker PostgreSQL). Integration tests catch more bugs than unit tests with mocked DB.",
		"kind": "pitfall",
		"confidence": 0.9,
		"tags": ["testing", "database", "best-practices"],
		"created": "2026-03-18T10:00:00Z",
		"updated": "2026-03-18T10:00:00Z",
		"source": "sync-20260318-100000-vwx234",
	},
]

for l in learnings:
	fm_dict = {k: v for k, v in l.items() if k != "body"}
	post = frontmatter.Post(l["body"], **fm_dict)
	path = memory_root / "learnings" / f"{l['id']}.md"
	path.write_text(frontmatter.dumps(post) + "\n", encoding="utf-8")
	print(f"  Learning: {path.name}")

# ---------------------------------------------------------------
# 4. Seed 2 session summaries (for cross-session analysis)
# ---------------------------------------------------------------
summaries_dir = memory_root / "summaries" / "20260325" / "120000"
summaries_dir.mkdir(parents=True, exist_ok=True)

summary_1 = {
	"id": "database-migration-setup",
	"title": "Database migration setup and PostgreSQL configuration",
	"created": "2026-03-20T10:00:00Z",
	"source": "sync-20260320-100000-aaa111",
	"description": "Set up database migrations with Alembic and configured PostgreSQL connection pooling.",
	"date": "2026-03-20",
	"time": "10:00:00",
	"coding_agent": "claude",
	"raw_trace_path": "/tmp/traces/session1.jsonl",
	"run_id": "sync-20260320-100000-aaa111",
	"repo_name": "test-project",
	"tags": ["database", "migration", "postgresql", "connection-pooling"],
}
summary_1_body = (
	"## User Intent\n\n"
	"Set up database migration infrastructure using Alembic with PostgreSQL. "
	"The user wanted to establish a reliable migration workflow that supports "
	"both development and production environments.\n\n"
	"## What Happened\n\n"
	"Configured Alembic for database migrations with PostgreSQL as the target database. "
	"Set up connection pooling with SQLAlchemy's QueuePool (pool_size=10, max_overflow=20). "
	"Created initial migration scripts for the core schema. Discussed best practices for "
	"migration versioning and rollback strategies."
)
post_1 = frontmatter.Post(summary_1_body, **summary_1)
s1_path = summaries_dir / "database-migration-setup.md"
s1_path.write_text(frontmatter.dumps(post_1) + "\n", encoding="utf-8")
print(f"  Summary 1: {s1_path.name}")

summary_2 = {
	"id": "database-indexing-optimization",
	"title": "Database indexing and query optimization",
	"created": "2026-03-22T14:00:00Z",
	"source": "sync-20260322-140000-bbb222",
	"description": "Optimized database queries with proper indexing and analyzed connection pooling performance.",
	"date": "2026-03-22",
	"time": "14:00:00",
	"coding_agent": "claude",
	"raw_trace_path": "/tmp/traces/session2.jsonl",
	"run_id": "sync-20260322-140000-bbb222",
	"repo_name": "test-project",
	"tags": ["database", "migration", "indexing", "connection-pooling", "performance"],
}
summary_2_body = (
	"## User Intent\n\n"
	"Optimize slow database queries and add proper indexing. The user noticed "
	"that some API endpoints were slow and suspected missing indexes on the "
	"database migration tables.\n\n"
	"## What Happened\n\n"
	"Analyzed slow queries using EXPLAIN ANALYZE and identified missing indexes on "
	"frequently queried columns. Added composite indexes for the session lookup pattern. "
	"Reviewed connection pooling metrics and found the pool was occasionally exhausted. "
	"Also discussed database migration best practices for adding indexes without downtime. "
	"Created a migration script to add the new indexes concurrently."
)
post_2 = frontmatter.Post(summary_2_body, **summary_2)
s2_path = summaries_dir / "database-indexing-optimization.md"
s2_path.write_text(frontmatter.dumps(post_2) + "\n", encoding="utf-8")
print(f"  Summary 2: {s2_path.name}")

# ---------------------------------------------------------------
# Print seed summary
# ---------------------------------------------------------------
print(f"\nSeeded files:")
print(f"  Decisions:  {len(decisions)}")
print(f"  Learnings:  {len(learnings)}")
print(f"  Summaries:  2")
print(f"  Total:      {len(decisions) + len(learnings) + 2}")

# ---------------------------------------------------------------
# 5. Run maintain
# ---------------------------------------------------------------
print("\n" + "=" * 60)
print("Running maintain (this makes real LLM calls)...")
print("=" * 60)

maintain_agent = LerimOAIAgent(default_cwd=str(tmp_base), config=cfg)
result = maintain_agent.maintain(
	memory_root=memory_root,
	workspace_root=workspace_root,
)

print("=" * 60)
print("Maintain completed!")
print(f"Cost: ${result.get('cost_usd', 0):.4f}")
print(f"\nCounts: {json.dumps(result.get('counts', {}), indent=2)}")

# ---------------------------------------------------------------
# 6. Print actions taken
# ---------------------------------------------------------------
actions_path = Path(result["artifacts"]["maintain_actions"])
if actions_path.exists():
	report = json.loads(actions_path.read_text(encoding="utf-8"))
	print(f"\nActions taken ({len(report.get('actions', []))})")
	for action in report.get("actions", []):
		print(f"  - {action.get('action', '?')}: {action.get('reason', '?')}")

	# Cross-session analysis results
	cross = report.get("cross_session_analysis", {})
	if cross:
		print(f"\nCross-session analysis:")
		signals = cross.get("signals", [])
		if signals:
			print(f"  Signals ({len(signals)}):")
			for s in signals:
				print(f"    - {s.get('topic', '?')} (in {s.get('summary_count', '?')} summaries): {s.get('recommendation', '?')}")
		contradictions = cross.get("contradictions", [])
		if contradictions:
			print(f"  Contradictions ({len(contradictions)}):")
			for c in contradictions:
				print(f"    - {c.get('memory_a', '?')} vs {c.get('memory_b', '?')}: {c.get('resolution', '?')}")
		gaps = cross.get("gaps", [])
		if gaps:
			print(f"  Gaps ({len(gaps)}):")
			for g in gaps:
				print(f"    - {g.get('topic', '?')}: {g.get('coverage', '?')}")
	else:
		print("\nCross-session analysis: not found in report")

	print(f"\nFull report:")
	print(json.dumps(report, indent=2))
else:
	print("\nWARNING: maintain_actions.json not found!")

# ---------------------------------------------------------------
# 7. Check hot-memory.md
# ---------------------------------------------------------------
hot_memory_path = lerim_dir / "hot-memory.md"
print(f"\nHot-memory.md exists: {hot_memory_path.exists()}")
if hot_memory_path.exists():
	print(f"Hot-memory.md size: {hot_memory_path.stat().st_size} bytes")

# ---------------------------------------------------------------
# 8. List all remaining memory files
# ---------------------------------------------------------------
print(f"\nRemaining memory files:")
for subdir in ("decisions", "learnings"):
	files = sorted((memory_root / subdir).glob("*.md"))
	print(f"  {subdir}/ ({len(files)} files):")
	for f in files:
		print(f"    - {f.name}")

for subdir in ("archived/decisions", "archived/learnings"):
	d = memory_root / subdir
	if d.exists():
		files = sorted(d.glob("*.md"))
		if files:
			print(f"  {subdir}/ ({len(files)} files):")
			for f in files:
				print(f"    - {f.name}")

print(f"\nRun folder: {result.get('run_folder', '')}")

## Test 3 — Hot-Memory Inspection

In [ ]:
# Read and display hot-memory.md contents. Check it's under ~2000 tokens.

hot_memory_path = lerim_dir / "hot-memory.md"

if hot_memory_path.exists():
	content = hot_memory_path.read_text(encoding="utf-8")
	print(f"Hot-memory.md ({len(content)} chars):")
	print("=" * 60)
	print(content)
	print("=" * 60)

	# Rough token estimate: ~4 chars per token for English text
	est_tokens = len(content) // 4
	print(f"\nEstimated tokens: ~{est_tokens}")
	if est_tokens <= 2500:
		print("Token budget: OK (under ~2000-2500 token target)")
	else:
		print(f"WARNING: Token budget exceeded ({est_tokens} >> 2000)")

	# Verify expected sections exist
	expected_sections = ["Active Decisions", "Key Learnings"]
	for section in expected_sections:
		if section in content:
			print(f"  Section '{section}': found")
		else:
			print(f"  Section '{section}': MISSING")
else:
	print("hot-memory.md was NOT created by the agent.")
	print("This may happen if the agent decided not to write it.")
	print(f"Expected path: {hot_memory_path}")

## Summary

In [ ]:
print("Maintain Agent Test Results")
print("=" * 40)
print(f"1. Prompt builder:    PASS")

if "result" in dir():
	counts = result.get("counts", {})
	total_actions = sum(counts.get(k, 0) for k in ("merged", "archived", "consolidated", "decayed"))
	print(f"2. Full maintain:     PASS ({total_actions} actions, {counts.get('unchanged', 0)} unchanged, ${result.get('cost_usd', 0):.4f})")
else:
	print(f"2. Full maintain:     SKIPPED (cell not run)")

if "hot_memory_path" in dir() and hot_memory_path.exists():
	print(f"3. Hot-memory:        PASS ({hot_memory_path.stat().st_size} bytes)")
else:
	print(f"3. Hot-memory:        NOT CREATED")

print()
print("Temp directory (inspect manually if needed):")
if "tmp_base" in dir():
	print(f"  {tmp_base}")